In [ ]:
import { useState, useEffect, useRef, useCallback } from "react";

// ── Constants ────────────────────────────────────────────────────────────────
const MAX_ATTEMPTS   = 5;
const LOCKOUT_SEC    = 30;
const WARN_AT        = 3;   // show warning banner after this many failures

// Demo credentials (in real system these would be hashed server-side)
const ACCOUNTS = {
  "admin":   "Admin@123",
  "indushree": "Secure#99",
  "guest":   "guest123",
};

const PHASES = { IDLE: "idle", SUCCESS: "success", LOCKED: "locked" };

function timestamp() {
  return new Date().toLocaleTimeString("en-IN", { hour12: false });
}

function shake(el) {
  if (!el) return;
  el.classList.remove("shake");
  void el.offsetWidth;
  el.classList.add("shake");
}

// ── Component ────────────────────────────────────────────────────────────────
export default function App() {
  const [username, setUsername]   = useState("");
  const [password, setPassword]   = useState("");
  const [showPw,   setShowPw]     = useState(false);
  const [phase,    setPhase]      = useState(PHASES.IDLE);
  const [attempts, setAttempts]   = useState(0);
  const [lockEnd,  setLockEnd]    = useState(null);   // Date
  const [countdown, setCountdown] = useState(0);
  const [log,      setLog]        = useState([]);
  const [shake_,   setShake_]     = useState(false);
  const [hint,     setHint]       = useState("");
  const cardRef  = useRef();
  const timerRef = useRef();

  // ── Countdown ticker ────────────────────────────────────────────────────
  useEffect(() => {
    if (phase !== PHASES.LOCKED || !lockEnd) return;
    timerRef.current = setInterval(() => {
      const left = Math.ceil((lockEnd - Date.now()) / 1000);
      if (left <= 0) {
        clearInterval(timerRef.current);
        setPhase(PHASES.IDLE);
        setAttempts(0);
        setCountdown(0);
        setLockEnd(null);
        addLog("system", "Lockout expired. Access restored.", "info");
      } else {
        setCountdown(left);
      }
    }, 250);
    return () => clearInterval(timerRef.current);
  }, [phase, lockEnd]);

  const addLog = useCallback((actor, msg, type = "fail") => {
    setLog(prev => [{ actor, msg, type, time: timestamp() }, ...prev].slice(0, 30));
  }, []);

  const triggerShake = () => {
    setShake_(true);
    setTimeout(() => setShake_(false), 600);
  };

  const handleLogin = () => {
    if (phase === PHASES.LOCKED) return;
    if (!username.trim() || !password.trim()) {
      setHint("Please enter both username and password.");
      triggerShake();
      return;
    }

    const expected = ACCOUNTS[username.toLowerCase()];
    if (expected && password === expected) {
      // ── SUCCESS ──────────────────────────────────────────────────────
      setPhase(PHASES.SUCCESS);
      setAttempts(0);
      setHint("");
      addLog(username, "Authentication successful.", "ok");
    } else {
      // ── FAILURE ──────────────────────────────────────────────────────
      const newAttempts = attempts + 1;
      setAttempts(newAttempts);
      triggerShake();

      const remaining = MAX_ATTEMPTS - newAttempts;

      if (newAttempts >= MAX_ATTEMPTS) {
        // Lock out
        const end = new Date(Date.now() + LOCKOUT_SEC * 1000);
        setLockEnd(end);
        setPhase(PHASES.LOCKED);
        setCountdown(LOCKOUT_SEC);
        setHint("");
        addLog(username || "unknown", `Account locked after ${MAX_ATTEMPTS} failed attempts.`, "lock");
      } else {
        const reason = !ACCOUNTS[username.toLowerCase()]
          ? "Unknown username."
          : "Wrong password.";
        setHint(
          newAttempts >= WARN_AT
            ? `⚠ ${reason} ${remaining} attempt${remaining !== 1 ? "s" : ""} remaining before lockout.`
            : `${reason} Please try again.`
        );
        addLog(username || "unknown", `Failed attempt ${newAttempts}/${MAX_ATTEMPTS}. ${reason}`, "fail");
      }
    }
    setPassword("");
  };

  const handleKeyDown = (e) => { if (e.key === "Enter") handleLogin(); };

  const resetAll = () => {
    clearInterval(timerRef.current);
    setUsername(""); setPassword(""); setPhase(PHASES.IDLE);
    setAttempts(0); setLockEnd(null); setCountdown(0); setHint("");
    addLog("system", "Session manually reset.", "info");
  };

  // ── Derived state ────────────────────────────────────────────────────────
  const pct        = (attempts / MAX_ATTEMPTS) * 100;
  const isLocked   = phase === PHASES.LOCKED;
  const isSuccess  = phase === PHASES.SUCCESS;
  const dangerZone = attempts >= WARN_AT && !isLocked && !isSuccess;

  // ── Render ───────────────────────────────────────────────────────────────
  return (
    <>
      <style>{CSS}</style>
      <div className="root">
        <div className="scanlines" aria-hidden />

        <div className="layout">
          {/* ── LEFT PANEL ── */}
          <div className="panel left-panel">
            <div className="terminal-header">
              <span className="dot red" /><span className="dot amber" /><span className="dot green" />
              <span className="terminal-title">AUTH_MONITOR.SYS</span>
            </div>

            <div className="brand">
              <div className={`shield-icon ${isLocked ? "locked" : isSuccess ? "unlocked" : ""}`}>
                {isLocked ? (
                  <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" strokeWidth="1.8" strokeLinecap="round" strokeLinejoin="round">
                    <rect x="3" y="11" width="18" height="11" rx="2"/>
                    <path d="M7 11V7a5 5 0 0 1 10 0v4"/>
                    <line x1="12" y1="15" x2="12" y2="17"/>
                  </svg>
                ) : isSuccess ? (
                  <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" strokeWidth="1.8" strokeLinecap="round" strokeLinejoin="round">
                    <path d="M12 22s8-4 8-10V5l-8-3-8 3v7c0 6 8 10 8 10z"/>
                    <polyline points="9 12 11 14 15 10"/>
                  </svg>
                ) : (
                  <svg viewBox="0 0 24 24" fill="none" stroke="currentColor" strokeWidth="1.8" strokeLinecap="round" strokeLinejoin="round">
                    <path d="M12 22s8-4 8-10V5l-8-3-8 3v7c0 6 8 10 8 10z"/>
                  </svg>
                )}
              </div>
              <h1 className="brand-title">SECURE<span>GATE</span></h1>
              <p className="brand-sub">Access Control Terminal v2.4</p>
            </div>

            {/* Attempt meter */}
            <div className="meter-block">
              <div className="meter-label">
                <span>ATTEMPT COUNTER</span>
                <span className={`meter-count ${dangerZone ? "danger" : ""}`}>
                  {attempts} / {MAX_ATTEMPTS}
                </span>
              </div>
              <div className="meter-track">
                <div
                  className={`meter-fill ${dangerZone ? "danger" : ""} ${isLocked ? "locked" : ""}`}
                  style={{ width: `${Math.min(pct, 100)}%` }}
                />
                {[1,2,3,4,5].map(i => (
                  <div key={i} className="meter-tick" style={{ left: `${(i/MAX_ATTEMPTS)*100}%` }} />
                ))}
              </div>
              <div className="meter-steps">
                {Array.from({ length: MAX_ATTEMPTS }).map((_, i) => (
                  <div key={i} className={`step-dot ${i < attempts ? (isLocked ? "locked" : "used") : ""}`} />
                ))}
              </div>
            </div>

            {/* Status chips */}
            <div className="chips">
              <div className={`chip ${isLocked ? "chip-red" : "chip-dim"}`}>
                {isLocked ? "🔴 LOCKED" : "⚪ LOCK OFF"}
              </div>
              <div className={`chip ${isSuccess ? "chip-green" : "chip-dim"}`}>
                {isSuccess ? "🟢 AUTHED" : "⚪ NO SESSION"}
              </div>
              <div className={`chip ${dangerZone ? "chip-amber" : "chip-dim"}`}>
                {dangerZone ? "🟡 WARNING" : "⚪ NORMAL"}
              </div>
            </div>

            {/* Hint for demo */}
            <div className="demo-hint">
              <div className="demo-hint-title">DEMO CREDENTIALS</div>
              <div className="demo-row"><span>admin</span><span>Admin@123</span></div>
              <div className="demo-row"><span>indushree</span><span>Secure#99</span></div>
              <div className="demo-row"><span>guest</span><span>guest123</span></div>
            </div>
          </div>

          {/* ── RIGHT PANEL ── */}
          <div className="panel right-panel">
            {/* Login form */}
            <div className={`form-card ${shake_ ? "shake" : ""} ${isLocked ? "form-locked" : ""} ${isSuccess ? "form-success" : ""}`} ref={cardRef}>

              {isSuccess ? (
                <div className="success-view">
                  <div className="success-icon">✓</div>
                  <h2 className="success-title">Access Granted</h2>
                  <p className="success-sub">Welcome back, <strong>{username}</strong></p>
                  <p className="success-detail">Session authenticated at {timestamp()}</p>
                  <button className="btn-logout" onClick={resetAll}>Sign Out</button>
                </div>
              ) : isLocked ? (
                <div className="locked-view">
                  <div className="lockout-icon">
                    <svg viewBox="0 0 24 24" fill="none" stroke="#ef4444" strokeWidth="2" strokeLinecap="round" strokeLinejoin="round" width="40" height="40">
                      <rect x="3" y="11" width="18" height="11" rx="2"/>
                      <path d="M7 11V7a5 5 0 0 1 10 0v4"/>
                    </svg>
                  </div>
                  <h2 className="lockout-title">Account Locked</h2>
                  <p className="lockout-msg">Too many failed attempts. Access is temporarily restricted.</p>
                  <div className="countdown-ring">
                    <svg viewBox="0 0 80 80" width="80" height="80">
                      <circle cx="40" cy="40" r="34" fill="none" stroke="#1e1e2e" strokeWidth="6"/>
                      <circle cx="40" cy="40" r="34" fill="none" stroke="#ef4444" strokeWidth="6"
                        strokeDasharray={`${2 * Math.PI * 34}`}
                        strokeDashoffset={`${2 * Math.PI * 34 * (1 - countdown / LOCKOUT_SEC)}`}
                        strokeLinecap="round"
                        transform="rotate(-90 40 40)"
                        style={{ transition: "stroke-dashoffset 0.25s linear" }}
                      />
                    </svg>
                    <span className="countdown-num">{countdown}s</span>
                  </div>
                  <p className="lockout-sub">Auto-unlock in {countdown} second{countdown !== 1 ? "s" : ""}</p>
                  <button className="btn-reset" onClick={resetAll}>Force Reset (Admin)</button>
                </div>
              ) : (
                <>
                  <div className="form-header">
                    <h2 className="form-title">Sign In</h2>
                    <p className="form-sub">Enter your credentials to continue</p>
                  </div>

                  {hint && (
                    <div className={`hint-box ${dangerZone ? "hint-warn" : "hint-err"}`}>
                      {hint}
                    </div>
                  )}

                  <div className="field">
                    <label className="field-label">USERNAME</label>
                    <div className="input-wrap">
                      <span className="input-icon">
                        <svg viewBox="0 0 24 24" width="16" height="16" fill="none" stroke="currentColor" strokeWidth="2" strokeLinecap="round" strokeLinejoin="round">
                          <path d="M20 21v-2a4 4 0 0 0-4-4H8a4 4 0 0 0-4 4v2"/>
                          <circle cx="12" cy="7" r="4"/>
                        </svg>
                      </span>
                      <input
                        className="input"
                        type="text"
                        value={username}
                        onChange={e => setUsername(e.target.value)}
                        onKeyDown={handleKeyDown}
                        placeholder="Enter username"
                        autoComplete="username"
                        disabled={isLocked}
                      />
                    </div>
                  </div>

                  <div className="field">
                    <label className="field-label">PASSWORD</label>
                    <div className="input-wrap">
                      <span className="input-icon">
                        <svg viewBox="0 0 24 24" width="16" height="16" fill="none" stroke="currentColor" strokeWidth="2" strokeLinecap="round" strokeLinejoin="round">
                          <rect x="3" y="11" width="18" height="11" rx="2"/>
                          <path d="M7 11V7a5 5 0 0 1 10 0v4"/>
                        </svg>
                      </span>
                      <input
                        className="input"
                        type={showPw ? "text" : "password"}
                        value={password}
                        onChange={e => setPassword(e.target.value)}
                        onKeyDown={handleKeyDown}
                        placeholder="Enter password"
                        autoComplete="current-password"
                        disabled={isLocked}
                      />
                      <button className="eye-btn" onClick={() => setShowPw(v => !v)} tabIndex={-1}>
                        {showPw ? "🙈" : "👁"}
                      </button>
                    </div>
                  </div>

                  <button className={`btn-login ${dangerZone ? "btn-warn" : ""}`} onClick={handleLogin}>
                    {dangerZone ? `⚠ Attempt Login (${MAX_ATTEMPTS - attempts} left)` : "Login →"}
                  </button>

                  {attempts > 0 && (
                    <button className="btn-clear" onClick={resetAll}>Clear & Reset</button>
                  )}
                </>
              )}
            </div>

            {/* ── Audit log ── */}
            <div className="log-panel">
              <div className="log-header">
                <span className="log-title">AUDIT LOG</span>
                <span className="log-count">{log.length} events</span>
              </div>
              <div className="log-body">
                {log.length === 0 ? (
                  <div className="log-empty">No events yet. Try logging in.</div>
                ) : log.map((e, i) => (
                  <div key={i} className={`log-row log-${e.type}`}>
                    <span className="log-time">{e.time}</span>
                    <span className={`log-badge badge-${e.type}`}>
                      {e.type === "ok" ? "AUTH" : e.type === "lock" ? "LOCK" : e.type === "info" ? "INFO" : "FAIL"}
                    </span>
                    <span className="log-actor">{e.actor}</span>
                    <span className="log-msg">{e.msg}</span>
                  </div>
                ))}
              </div>
            </div>
          </div>
        </div>
      </div>
    </>
  );
}

// ── CSS ──────────────────────────────────────────────────────────────────────
const CSS = `
  @import url('https://fonts.googleapis.com/css2?family=Share+Tech+Mono&family=Plus+Jakarta+Sans:wght@400;500;600;700;800&display=swap');

  *, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }

  .root {
    min-height: 100vh;
    background: linear-gradient(135deg, #eef2ff 0%, #f0fdf4 50%, #fefce8 100%);
    font-family: 'Plus Jakarta Sans', sans-serif;
    display: flex;
    align-items: center;
    justify-content: center;
    padding: 24px 16px;
    position: relative;
    overflow: hidden;
  }

  .scanlines {
    position: fixed; inset: 0;
    background-image: radial-gradient(circle at 20% 20%, rgba(99,102,241,0.06) 0%, transparent 50%),
                      radial-gradient(circle at 80% 80%, rgba(34,197,94,0.05) 0%, transparent 50%);
    pointer-events: none; z-index: 0;
  }

  .layout {
    position: relative; z-index: 1;
    display: flex; gap: 20px;
    width: 100%; max-width: 960px;
    align-items: flex-start;
  }

  .panel {
    border-radius: 16px;
    border: 1px solid rgba(99,102,241,0.15);
    background: #ffffff;
    overflow: hidden;
    box-shadow: 0 4px 24px rgba(99,102,241,0.08), 0 1px 4px rgba(0,0,0,0.04);
  }

  /* ── LEFT PANEL ── */
  .left-panel {
    width: 280px; flex-shrink: 0;
    padding: 0 0 20px;
  }

  .terminal-header {
    background: #f8f8ff;
    border-bottom: 1px solid #e8e8f8;
    padding: 10px 16px;
    display: flex; align-items: center; gap: 6px;
    font-family: 'Share Tech Mono', monospace;
    font-size: 10px; color: #a0a0c8;
    letter-spacing: 0.1em;
  }
  .dot { width: 10px; height: 10px; border-radius: 50%; }
  .dot.red    { background: #ff5f57; }
  .dot.amber  { background: #febc2e; }
  .dot.green  { background: #28c840; }
  .terminal-title { margin-left: 8px; }

  .brand { padding: 24px 20px 16px; text-align: center; }

  .shield-icon {
    width: 60px; height: 60px; margin: 0 auto 14px;
    border-radius: 50%;
    border: 2px solid #e0e0f8;
    background: #f5f5ff;
    display: flex; align-items: center; justify-content: center;
    color: #a5a8f8;
    transition: all 0.4s;
  }
  .shield-icon svg { width: 28px; height: 28px; }
  .shield-icon.locked { border-color: #fca5a5; color: #ef4444; background: #fef2f2; box-shadow: 0 0 16px rgba(239,68,68,0.15); }
  .shield-icon.unlocked { border-color: #86efac; color: #16a34a; background: #f0fdf4; box-shadow: 0 0 16px rgba(34,197,94,0.15); }

  .brand-title {
    font-size: 20px; font-weight: 800;
    color: #374151; letter-spacing: 0.12em;
  }
  .brand-title span { color: #6366f1; }
  .brand-sub { font-size: 10px; color: #9ca3af; letter-spacing: 0.08em; margin-top: 4px; font-family: 'Share Tech Mono', monospace; }

  .meter-block { padding: 0 20px 16px; }
  .meter-label { display: flex; justify-content: space-between; font-size: 10px; color: #9ca3af; letter-spacing: 0.08em; margin-bottom: 8px; font-family: 'Share Tech Mono', monospace; }
  .meter-count { color: #6366f1; font-weight: 700; }
  .meter-count.danger { color: #d97706; }

  .meter-track {
    height: 6px; background: #f1f1fb; border-radius: 3px;
    position: relative; overflow: hidden;
    border: 1px solid #e5e5f5;
  }
  .meter-fill {
    height: 100%; border-radius: 3px;
    background: linear-gradient(90deg, #818cf8, #6366f1);
    transition: width 0.4s ease, background 0.4s;
  }
  .meter-fill.danger { background: linear-gradient(90deg, #fbbf24, #f59e0b); }
  .meter-fill.locked { background: linear-gradient(90deg, #f87171, #ef4444); }
  .meter-tick { position: absolute; top: 0; bottom: 0; width: 1px; background: #fff; transform: translateX(-50%); }

  .meter-steps { display: flex; justify-content: space-between; padding: 0 2px; margin-top: 8px; }
  .step-dot { width: 10px; height: 10px; border-radius: 50%; background: #f1f1fb; border: 1.5px solid #ddd8f8; transition: all 0.3s; }
  .step-dot.used { background: #6366f1; border-color: #6366f1; box-shadow: 0 0 6px rgba(99,102,241,0.35); }
  .step-dot.locked { background: #ef4444; border-color: #ef4444; box-shadow: 0 0 6px rgba(239,68,68,0.35); }

  .chips { padding: 0 20px 16px; display: flex; flex-direction: column; gap: 6px; }
  .chip { padding: 6px 10px; border-radius: 6px; font-size: 10px; font-weight: 700; letter-spacing: 0.08em; font-family: 'Share Tech Mono', monospace; border: 1px solid transparent; transition: all 0.3s; }
  .chip-dim  { background: #f8f8ff; border-color: #e8e8f8; color: #c0c0d8; }
  .chip-red  { background: #fef2f2; border-color: #fca5a5; color: #dc2626; }
  .chip-green{ background: #f0fdf4; border-color: #86efac; color: #16a34a; }
  .chip-amber{ background: #fffbeb; border-color: #fcd34d; color: #b45309; }

  .demo-hint { margin: 0 20px; padding: 12px; background: #f8f8ff; border: 1px solid #e8e8f8; border-radius: 8px; }
  .demo-hint-title { font-size: 9px; font-weight: 700; color: #a0a0c0; letter-spacing: 0.15em; margin-bottom: 8px; font-family: 'Share Tech Mono', monospace; }
  .demo-row { display: flex; justify-content: space-between; font-size: 11px; color: #9ca3af; font-family: 'Share Tech Mono', monospace; padding: 3px 0; border-bottom: 1px solid #ededf8; }
  .demo-row:last-child { border-bottom: none; }
  .demo-row span:first-child { color: #6366f1; font-weight: 600; }

  /* ── RIGHT PANEL ── */
  .right-panel { flex: 1; padding: 24px; display: flex; flex-direction: column; gap: 18px; }

  .form-card {
    background: #ffffff;
    border: 1.5px solid #e8e8f8;
    border-radius: 12px;
    padding: 28px 24px;
    transition: border-color 0.3s, box-shadow 0.3s;
  }
  .form-card.form-locked { border-color: #fca5a5; box-shadow: 0 0 24px rgba(239,68,68,0.08); }
  .form-card.form-success { border-color: #86efac; box-shadow: 0 0 24px rgba(34,197,94,0.08); }

  @keyframes shake-anim {
    0%,100%{transform:translateX(0)} 15%{transform:translateX(-8px)} 30%{transform:translateX(8px)}
    45%{transform:translateX(-6px)} 60%{transform:translateX(6px)} 75%{transform:translateX(-4px)} 90%{transform:translateX(4px)}
  }
  .shake { animation: shake-anim 0.55s ease; }

  .form-header { margin-bottom: 20px; }
  .form-title { font-size: 22px; font-weight: 800; color: #1e1b4b; letter-spacing: -0.3px; }
  .form-sub { font-size: 12px; color: #9ca3af; margin-top: 4px; }

  .hint-box {
    padding: 10px 14px; border-radius: 8px; font-size: 12px;
    margin-bottom: 18px; font-weight: 500; line-height: 1.4;
    font-family: 'Share Tech Mono', monospace;
  }
  .hint-err  { background: #fef2f2; border: 1px solid #fca5a5; color: #dc2626; }
  .hint-warn { background: #fffbeb; border: 1px solid #fcd34d; color: #b45309; }

  .field { margin-bottom: 16px; }
  .field-label { display: block; font-size: 10px; font-weight: 700; color: #6366f1; letter-spacing: 0.12em; margin-bottom: 7px; font-family: 'Share Tech Mono', monospace; }
  .input-wrap { position: relative; display: flex; align-items: center; }
  .input-icon { position: absolute; left: 12px; color: #c4c4e0; pointer-events: none; }
  .input {
    width: 100%; background: #f8f8ff; border: 1.5px solid #e0e0f5; border-radius: 8px;
    padding: 11px 40px 11px 38px; color: #1e1b4b;
    font-family: 'Plus Jakarta Sans', sans-serif; font-size: 14px; outline: none;
    transition: border-color 0.2s, box-shadow 0.2s;
  }
  .input:focus { border-color: #6366f1; background: #fff; box-shadow: 0 0 0 3px rgba(99,102,241,0.1); }
  .input::placeholder { color: #c4c4e0; }
  .eye-btn { position: absolute; right: 10px; background: none; border: none; cursor: pointer; font-size: 14px; padding: 4px; }

  .btn-login {
    width: 100%; padding: 13px;
    background: linear-gradient(135deg, #6366f1 0%, #4f46e5 100%);
    border: none; border-radius: 8px;
    color: #fff; font-family: 'Plus Jakarta Sans', sans-serif;
    font-size: 14px; font-weight: 700; cursor: pointer;
    letter-spacing: 0.03em; margin-top: 4px;
    box-shadow: 0 4px 14px rgba(99,102,241,0.35);
    transition: opacity 0.2s, transform 0.1s;
  }
  .btn-login:hover { opacity: 0.92; transform: translateY(-1px); }
  .btn-login.btn-warn {
    background: linear-gradient(135deg, #f59e0b 0%, #d97706 100%);
    box-shadow: 0 4px 14px rgba(245,158,11,0.35);
  }

  .btn-clear {
    width: 100%; margin-top: 8px; padding: 9px;
    background: transparent; border: 1.5px solid #e8e8f8;
    border-radius: 8px; color: #9ca3af;
    font-family: 'Plus Jakarta Sans', sans-serif; font-size: 12px;
    cursor: pointer; transition: border-color 0.2s, color 0.2s;
  }
  .btn-clear:hover { border-color: #c4c4e0; color: #6b7280; }

  /* ── SUCCESS VIEW ── */
  .success-view { text-align: center; padding: 16px 0; }
  .success-icon {
    width: 64px; height: 64px; border-radius: 50%; margin: 0 auto 16px;
    background: #f0fdf4; border: 2px solid #86efac;
    display: flex; align-items: center; justify-content: center;
    font-size: 28px; color: #16a34a;
    animation: pop 0.4s cubic-bezier(0.34,1.56,0.64,1);
  }
  @keyframes pop { from { transform: scale(0.5); opacity: 0; } to { transform: scale(1); opacity: 1; } }
  .success-title { font-size: 22px; font-weight: 800; color: #15803d; margin-bottom: 6px; }
  .success-sub { font-size: 14px; color: #6b7280; }
  .success-detail { font-size: 11px; color: #9ca3af; font-family: 'Share Tech Mono', monospace; margin: 8px 0 20px; }
  .btn-logout {
    padding: 10px 28px; background: transparent; border: 1.5px solid #86efac;
    border-radius: 8px; color: #16a34a; font-family: 'Plus Jakarta Sans', sans-serif;
    font-size: 13px; font-weight: 600; cursor: pointer; transition: background 0.2s;
  }
  .btn-logout:hover { background: #f0fdf4; }

  /* ── LOCKED VIEW ── */
  .locked-view { text-align: center; padding: 8px 0; }
  .lockout-icon { margin-bottom: 12px; }
  .lockout-title { font-size: 20px; font-weight: 800; color: #dc2626; margin-bottom: 8px; }
  .lockout-msg { font-size: 12px; color: #9ca3af; margin-bottom: 20px; line-height: 1.5; }
  .countdown-ring { position: relative; width: 80px; height: 80px; margin: 0 auto 10px; display: flex; align-items: center; justify-content: center; }
  .countdown-num { position: absolute; font-family: 'Share Tech Mono', monospace; font-size: 16px; font-weight: 700; color: #dc2626; }
  .lockout-sub { font-size: 12px; color: #9ca3af; font-family: 'Share Tech Mono', monospace; margin-bottom: 18px; }
  .btn-reset {
    padding: 8px 20px; background: transparent; border: 1.5px solid #fca5a5;
    border-radius: 8px; color: #9ca3af; font-family: 'Plus Jakarta Sans', sans-serif;
    font-size: 11px; cursor: pointer; transition: all 0.2s;
  }
  .btn-reset:hover { border-color: #ef4444; color: #dc2626; background: #fef2f2; }

  /* ── AUDIT LOG ── */
  .log-panel { background: #fff; border: 1.5px solid #e8e8f8; border-radius: 12px; overflow: hidden; }
  .log-header {
    padding: 10px 16px; background: #f8f8ff; border-bottom: 1px solid #e8e8f8;
    display: flex; justify-content: space-between; align-items: center;
  }
  .log-title { font-size: 10px; font-weight: 700; color: #6366f1; letter-spacing: 0.15em; font-family: 'Share Tech Mono', monospace; }
  .log-count { font-size: 10px; color: #c4c4e0; font-family: 'Share Tech Mono', monospace; }
  .log-body { max-height: 190px; overflow-y: auto; padding: 6px 0; }
  .log-empty { padding: 20px; text-align: center; font-size: 11px; color: #d0d0e8; font-family: 'Share Tech Mono', monospace; }
  .log-row {
    display: grid; grid-template-columns: 52px 40px 80px 1fr;
    gap: 8px; align-items: center;
    padding: 6px 14px; font-size: 11px;
    border-bottom: 1px solid #f5f5fb;
    font-family: 'Share Tech Mono', monospace;
    transition: background 0.15s;
  }
  .log-row:hover { background: #f8f8ff; }
  .log-time  { color: #c4c4e0; }
  .log-actor { color: #6366f1; overflow: hidden; text-overflow: ellipsis; white-space: nowrap; font-weight: 600; }
  .log-msg   { color: #9ca3af; font-size: 10px; }

  .log-badge { padding: 2px 5px; border-radius: 4px; font-size: 9px; font-weight: 700; letter-spacing: 0.05em; text-align: center; }
  .badge-ok   { background: #dcfce7; color: #15803d; }
  .badge-fail { background: #fee2e2; color: #dc2626; }
  .badge-lock { background: #fee2e2; color: #b91c1c; }
  .badge-info { background: #ede9fe; color: #6366f1; }

  /* Scrollbar */
  .log-body::-webkit-scrollbar { width: 4px; }
  .log-body::-webkit-scrollbar-track { background: transparent; }
  .log-body::-webkit-scrollbar-thumb { background: #e0e0f5; border-radius: 2px; }

  @media (max-width: 680px) {
    .layout { flex-direction: column; }
    .left-panel { width: 100%; }
  }
`;
